In [13]:
import wandb
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AutoTokenizer, AutoModelForCausalLM
# Uncomment this for Mistral models when using GPU
#from unsloth import FastLanguageModel 

# Model selection - change this to the model you want to use
MODEL_TYPE = "gpt2"  # Options: "gpt2", "mistral" 
MODEL_ARTIFACT = "master_thesis_math_lm/gpt2-math-instruct/gpt2-math-model-finetuned:v6"  # Change as needed

# Initialize a W&B run
run = wandb.init()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: WARNING Unable to render HTML, can't import display from ipython.core


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


CommError: type model specified but this artifact is of type wandb-history

In [ ]:
####### MODELS
# 'master_thesis_math_lm/gpt2-math/gpt2-math-model:v0' ----------------------- (17.03.25) Pre-trained
# 'master_thesis_math_lm/gpt2-math/gpt2-math-sft-final:v0' ------------------- (20.03.25) Curriculum learning
# 'master_thesis_math_lm/gpt2-math-instruct/gpt2-math-model-finetuned:v4' ---- (24.04.25) 1st draft instruction learning
# 'master_thesis_math_lm/gpt2-math-instruct/gpt2-math-model-finetuned:v6' ---- (25.04.25) full run instruction learning
# 'master_thesis_math_lm/mistral-math-final/mistral-7b-math-unsloth-model:v0'- (30.03.25) Mistral pre-trained
# 'master_thesis_math_lm/gpt2-large-cl-final/run-r6yu8439-history:v0' -------- (02.04.25) CL crashed

In [ ]:
# Use this cell for GPT-2 models
if MODEL_TYPE == "gpt2":
    # Use the specified artifact
    artifact = run.use_artifact(MODEL_ARTIFACT, type='model')
    # Download the artifact and get the directory path
    artifact_dir = artifact.download()
    
    # Load the tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained(artifact_dir)
    model = GPT2LMHeadModel.from_pretrained(artifact_dir)
    
    # Set pad token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    
    # Move to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"GPT-2 model loaded and moved to {device}")

In [ ]:
# Use this cell for Mistral models with Unsloth
if MODEL_TYPE == "mistral":
    # Only uncomment and run this cell when using Mistral models with a GPU
    
    # Use the specified artifact
    artifact = run.use_artifact(MODEL_ARTIFACT, type='model')
    # Download the artifact and get the directory path
    artifact_dir = artifact.download()
    
    # Load with Unsloth (for GPU)
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        artifact_dir,
        max_seq_length=2048,  # Adjust as needed
        dtype=torch.bfloat16,  # Or whatever precision you need
        load_in_4bit=True,     # Optional, for quantization
    )
    
    print("Mistral model loaded with Unsloth")

In [ ]:
# Alternative loading for Mistral without Unsloth
if MODEL_TYPE == "mistral" and 'model' not in locals():
    # Use the specified artifact
    artifact = run.use_artifact(MODEL_ARTIFACT, type='model')
    # Download the artifact and get the directory path
    artifact_dir = artifact.download()
    
    # Load the model and tokenizer using standard methods
    tokenizer = AutoTokenizer.from_pretrained(artifact_dir)
    model = AutoModelForCausalLM.from_pretrained(
        artifact_dir,
        device_map="auto",  # Automatically distribute model across available GPUs
        torch_dtype=torch.float16  # Use float16 precision to save memory
    )
    
    # Set pad token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id
    
    print("Mistral model loaded using standard method")

In [ ]:
# Example: Generate text using the model
prompt = 'Jasper has 5 apples and eats 2 of them. How many apples does he have left?'

# Process the prompt
inputs = tokenizer(prompt, return_tensors='pt', padding=True)

# Move input tensors to the same device as the model
if MODEL_TYPE == "gpt2":
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate text
output = model.generate(
    **inputs,
    max_length=150,
    num_return_sequences=1,
    repetition_penalty=1.2,
    temperature=0.7,
    pad_token_id=tokenizer.pad_token_id,
    do_sample=True,
)

# Decode and print the generated text
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print("\nPrompt:", prompt)
print("\nFull output:", generated_text)

# For GPT-2 models, the prompt is repeated in the output, so we can extract just the response
if MODEL_TYPE == "gpt2":
    response = generated_text[len(prompt):].strip()
    print("\nJust the response:", response)

In [ ]:
# Finish the W&B run
wandb.finish()